# V-GLLVM generative check — synthetic patients vs the real data

How good is the variational GLLVM as a **generative model**? We draw synthetic patients from the fitted
8-factor map and compare their **raw-variable distributions** to the observed data.

Generative recipe: `f ~ N(0, Φ)` → `η = α + Λ f` → `x_j ~ F_j(η_j)` per item, inverting the
rank-INT copula back to the raw clinical scale for continuous items. We fit a `covariate_mode="none"`
variant so the copula inverts exactly (the canonical map is covariate-adjusted, which deliberately removes
covariate effects from the marginals). Companion script: `notebooks/run_gllvm_synthetic_check.py`.

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

REPO = next(p for p in Path.cwd().resolve().parents if (p / 'pyproject.toml').exists()) \
    if not (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().resolve()
SRC = REPO / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from face.models.variational.generative import (
    correlation_block,
    generate_synthetic,
    marginal_summary,
)
from face.models.variational.gllvm_model_oop import GLLVMConfig, GLLVMRunner

print('repo:', REPO)

## 1. Fit (or load) the generative model

`covariate_mode="none"` + low-rank `q`. If the script has already run, this loads the cached fit instantly.

In [ ]:
cfg = replace(GLLVMConfig(), covariate_mode='none', include_covariates=False, q_rank=2,
              epochs=2500, output_dir=REPO / 'results' / 'face' / 'gllvm_oop_gen')
runner = GLLVMRunner(cfg)
fit = runner.run_plan()          # cached if the script ran; else fits (~5 min on CPU)
data, model = fit['data'], fit['model']
print('stage', fit['stage'], '| items', len(data.items), '| copula items', len(data.copula))

## 2. Draw synthetic patients (raw clinical scale)

In [ ]:
baseline = pd.read_parquet(cfg.processed_dir / 'baseline_v0.parquet')
observed = baseline[[c for c in data.items if c in baseline.columns]]
families = {it: data.families[i] for i, it in enumerate(data.items)}

synth = generate_synthetic(model, data, n=len(data.index), seed=20260606)
print('synthetic patients:', synth.shape)
synth.head()

## 3. Marginal fidelity — observed vs synthetic per variable

Kolmogorov–Smirnov distance per indicator (0 = identical). Smaller is better.

In [ ]:
ms = marginal_summary(observed, synth, families)
print('KS median  :', round(ms.ks.median(), 3))
print('KS by family:', ms.groupby('family').ks.median().round(3).to_dict())
print('\nworst-fit variables:')
display(ms.head(8)[['item', 'family', 'obs_mean', 'syn_mean', 'obs_sd', 'syn_sd', 'ks']])

In [ ]:
PANEL = ['egf', 'cgi01', 'bmi', 'hba1c', 'crp', 'wbc', 'psqi', 'cvlt_total_recall',
         'tmt_a_time_sec', 'altman', 'isf01', 'suoccur_alcool']
items = [it for it in PANEL if it in synth.columns and it in observed.columns]
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, it in zip(axes.ravel(), items):
    o = pd.to_numeric(observed[it], errors='coerce').dropna().to_numpy()
    s = synth[it].dropna().to_numpy()
    if families[it] in ('bernoulli', 'ordinal', 'count') and np.union1d(o, s).size <= 12:
        vals = np.union1d(np.unique(o), np.unique(s)); w = 0.4
        ax.bar(vals - w/2, [np.mean(o == v) for v in vals], width=w, color='#4878a8', alpha=0.85, label='observed')
        ax.bar(vals + w/2, [np.mean(s == v) for v in vals], width=w, color='#e76f51', alpha=0.85, label='synthetic')
    else:
        lo, hi = np.nanpercentile(np.concatenate([o, s]), [0.5, 99.5])
        bins = np.linspace(lo, hi, 40)
        ax.hist(o, bins=bins, density=True, color='#4878a8', alpha=0.55, label='observed')
        ax.hist(s, bins=bins, density=True, color='#e76f51', alpha=0.55, label='synthetic')
    ax.set_title(it, fontsize=10); ax.tick_params(labelsize=7)
axes.ravel()[0].legend(fontsize=8)
fig.suptitle('Observed (blue) vs synthetic (orange) raw distributions', fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.97)); plt.show()

## 4. Joint structure — does the factor model reproduce the correlations?

Marginals are reproduced near-exactly by the copula inversion; the real test of the *latent factor model*
is whether it reproduces the **pairwise correlation structure**. SRMR = root-mean-square off-diagonal
difference (smaller is better).

In [ ]:
cont = [it for i, it in enumerate(data.items) if data.families[i] == 'gaussian']
co, cs, srmr = correlation_block(observed, synth, cont)
print('continuous-block correlation SRMR:', round(srmr, 3))
off = ~np.eye(len(co), dtype=bool)
x, y = co.to_numpy()[off], cs.to_numpy()[off]
keep = np.isfinite(x) & np.isfinite(y)
print('corr(observed, synthetic off-diagonal):', round(float(np.corrcoef(x[keep], y[keep])[0, 1]), 3))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (mat, title, vmax) in zip(axes, [(co, 'observed', 1.0), (cs, 'synthetic', 1.0), (co - cs, 'obs - syn', 0.4)]):
    im = ax.imshow(mat.to_numpy(), vmin=-vmax, vmax=vmax, cmap='RdBu_r')
    ax.set_title(title, fontsize=11); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle(f'Continuous-block correlations · SRMR = {srmr:.3f}', fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.95)); plt.show()

## Takeaway

The marginals are reproduced near-exactly (copula inversion is the inverse rank-INT). The correlation
SRMR measures how faithfully the **8-factor latent structure** reproduces the joint distribution — a low
SRMR means the map is a good generative model of the data, not just of each variable in isolation.
Honest caveat: this is the `covariate_mode="none"` generative variant; the canonical map is covariate-
adjusted (it removes age/sex/edu/site effects from the marginals by design).